# Deep Hedging

A hedging policy is a neural network mapping market state to position size, trained by stochastic gradient descent on a convex risk measure of terminal profit and loss over simulated paths. Classical delta hedging assumes frictionless complete markets and derives the hedge analytically. Deep hedging drops both assumptions. Transaction costs, discrete rebalancing, and unhedgeable state enter the simulator, and the optimiser finds the policy the market actually rewards.

This notebook trains and evaluates the framework end to end. Sections cover path simulation with exact replay, the risk objective, training under proportional costs, comparison against the Black-Scholes delta baseline, stochastic volatility with variance-aware features, barrier liabilities, and deep BSDE pricing. Every quantitative claim in the library is pinned by a test against a closed form or a statistical relationship. The notebook shows the same checks interactively.

In [ ]:
import matplotlib.pyplot as plt
import torch

from deephedging import (
    BSDEConfig,
    BSDEProblem,
    CVaR,
    DeepBSDESolver,
    EuropeanCall,
    FeedForwardPolicy,
    GBMSimulator,
    HestonSimulator,
    NoiseSpec,
    ProportionalCost,
    RunningMaxFeatures,
    TrainConfig,
    UpAndOutCall,
    VarianceFeatures,
    bs_call_price,
    delta_hedge_positions,
    hedge_pnl,
    pnl_from_positions,
    pnl_summary,
    train,
    train_bsde,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device {DEVICE}")

## Market simulation

Paths are generated on the fly each training batch, so data is unbounded and nothing overfits a stored dataset. The GBM sampler is exact in distribution. It evolves the log price, which keeps floating point error growth at the square root of the step count and avoids the biased increment dropping of a multiplicative recursion in reduced precision.

A `NoiseSpec` pair `(seed, stream)` names every draw. The same pair reproduces the same paths bit for bit on a given backend, each training iteration owns its own child stream, and the planned CUDA generator consumes the pair as its Philox subsequence. Replay of any single batch is therefore exact.

In [ ]:
SIGMA, MATURITY, STRIKE, N_STEPS = 0.2, 0.25, 100.0, 30
sim = GBMSimulator(s0=100.0, sigma=SIGMA, maturity=MATURITY, n_steps=N_STEPS, device=DEVICE)

state = sim.simulate(50, noise=NoiseSpec(seed=7))
replay = sim.simulate(50, noise=NoiseSpec(seed=7))
assert torch.equal(state.spot, replay.spot)

grid = torch.linspace(0.0, MATURITY, N_STEPS + 1)
plt.figure(figsize=(7, 3))
plt.plot(grid, state.spot.cpu(), linewidth=0.7, alpha=0.7)
plt.xlabel("time (years)")
plt.ylabel("spot")
plt.title("GBM paths, exact replay verified")
plt.tight_layout()

## Liability, frictions, objective

The desk sells a European call, collects the Black-Scholes premium, and hedges in the underlying. Each trade costs a fixed fraction of traded notional. Terminal PnL per path is premium plus trading gains minus costs minus the payoff.

The training objective is conditional value at risk of the loss via the Rockafellar-Uryasev form. Minimising `w + E[(L - w)+] / (1 - alpha)` jointly over the threshold `w` and the policy equals minimising CVaR. The identity is pointwise in the policy, so a nonconvex network does not break it. The threshold is a learned parameter warm-started at the empirical quantile, never a per-batch quantile, because per-batch inner minimisation is optimistically biased by Jensen and a parameter stays synchronised under distributed training. Only the worst `(1 - alpha)` fraction of paths carries gradient, so batch sizes scale with `1 / (1 - alpha)`.

In [ ]:
payoff = EuropeanCall(strike=STRIKE)
cost = ProportionalCost(rate=2e-3)
PREMIUM = float(bs_call_price(100.0, STRIKE, SIGMA, MATURITY))
print(f"premium {PREMIUM:.4f}")

## Training

The episode engine runs a sequential time-major loop. At each rebalancing date the feature map builds the observation, the policy emits a position, gains and costs accrue. The default observation is log moneyness, time to maturity, and current position, which is the Markovian sufficient state for vanilla liabilities under GBM.

In [ ]:
torch.manual_seed(11)
policy = FeedForwardPolicy(hidden_sizes=(64, 64)).to(DEVICE)
config = TrainConfig(n_iterations=600, batch_paths=16384, lr=1e-3, seed=4)
result = train(sim, policy, payoff, cost, CVaR(alpha=0.95), config, premium=PREMIUM)

plt.figure(figsize=(7, 3))
plt.plot(result.losses, linewidth=0.8)
plt.xlabel("iteration")
plt.ylabel("CVaR objective")
plt.title("training loss")
plt.tight_layout()

## Evaluation against baselines

Three books on identical out-of-sample paths. No hedge keeps the premium and eats the payoff. The Black-Scholes delta hedge is optimal in the frictionless continuous limit but pays costs for every rebalance and ignores them when choosing positions. The trained policy knows the costs exist.

Expected shortfall here is the evaluation estimator, clamped at the sample maximum because quantile interpolation plus tail rescaling can otherwise exceed the worst observed loss at extreme levels. Keep it out of training losses.

In [ ]:
eval_state = sim.simulate(100_000, noise=NoiseSpec(seed=99))
with torch.no_grad():
    deep_pnl = hedge_pnl(eval_state, policy, payoff, cost, premium=PREMIUM)
deltas = delta_hedge_positions(eval_state.spot, STRIKE, SIGMA, MATURITY)
delta_pnl = pnl_from_positions(eval_state, deltas, payoff, cost, premium=PREMIUM)
naked_pnl = PREMIUM - payoff(eval_state.spot)

for name, pnl in (("no hedge", naked_pnl), ("BS delta", delta_pnl), ("deep", deep_pnl)):
    print(f"{name:9s} {pnl_summary(pnl)}")

plt.figure(figsize=(7, 3))
for name, pnl in (("BS delta", delta_pnl), ("deep", deep_pnl)):
    plt.hist(pnl.cpu().numpy(), bins=200, range=(-4, 3), alpha=0.5, label=name, density=True)
plt.xlabel("terminal PnL")
plt.legend()
plt.title("hedged PnL under proportional costs")
plt.tight_layout()

The delta hedge trades aggressively because nothing charges it for turnover, so costs eat its premium and widen its left tail. The trained policy under-hedges relative to delta and earns a tighter loss tail at the chosen confidence level. In the frictionless limit the two coincide, which the test suite checks by a dispersion bound on the discrete delta hedge.

## Stochastic volatility

Heston dynamics make variance a second state variable. The simulator uses the full-truncation Euler scheme, the standard bias-minimising discretisation when the Feller condition fails, and exposes the clamped variance path as a named channel on the market state. `VarianceFeatures` appends it to the observation, so the policy can distinguish calm from stressed regimes at the same spot level. A spot-only policy cannot represent that distinction.

In [ ]:
heston = HestonSimulator(
    s0=100.0, v0=0.04, kappa=1.5, theta=0.04, xi=0.5, rho=-0.7,
    maturity=MATURITY, n_steps=N_STEPS, device=DEVICE,
)
torch.manual_seed(12)
vol_policy = FeedForwardPolicy(n_features=4, hidden_sizes=(64, 64)).to(DEVICE)
vol_config = TrainConfig(n_iterations=600, batch_paths=16384, lr=1e-3, seed=5)
vol_result = train(
    heston, vol_policy, payoff, cost, CVaR(alpha=0.95), vol_config,
    premium=PREMIUM, feature_map=VarianceFeatures(),
)

heston_eval = heston.simulate(100_000, noise=NoiseSpec(seed=98))
with torch.no_grad():
    vol_pnl = hedge_pnl(
        heston_eval, vol_policy, payoff, cost, premium=PREMIUM, feature_map=VarianceFeatures()
    )
print(f"heston variance-aware {pnl_summary(vol_pnl)}")

## Barrier liabilities

An up-and-out call knocks out when the spot touches the barrier at any monitoring date, inception included. The running maximum is the sufficient path statistic, cached on the market state as a cumulative maximum so the episode pays linear rather than quadratic cost in the horizon. `RunningMaxFeatures` feeds it to the policy, which can learn to stop hedging a dead contract.

In [ ]:
barrier_payoff = UpAndOutCall(strike=STRIKE, barrier=115.0)
mc_state = sim.simulate(200_000, noise=NoiseSpec(seed=97))
vanilla_price = float(payoff(mc_state.spot).mean())
barrier_price = float(barrier_payoff(mc_state.spot).mean())
print(f"monte carlo price  vanilla {vanilla_price:.4f}  up-and-out {barrier_price:.4f}")

features = RunningMaxFeatures()(mc_state, N_STEPS // 2, mc_state.spot.new_tensor(0.5),
                                mc_state.spot.new_zeros(mc_state.n_paths))
print(f"observation width {features.shape[1]} at mid-episode")

## Deep BSDE pricing

High-dimensional semilinear pricing PDEs reformulate as backward SDEs. The solver learns the inception value `Y0` and a network for the volatility process `Z`, evolves `Y` forward by explicit Euler, and minimises the terminal mismatch against the payoff. For a generator uniformly Lipschitz in `(Y, Z)` the achieved loss is an a posteriori bound on the solution error, so the final loss certifies accuracy rather than merely indicating convergence. Scope is the semilinear Lipschitz class. Proportional costs, gamma constraints, and uncertain volatility leave it and need reflected or second-order BSDE machinery.

With the zero generator the solved `Y0` is the plain expectation, here the undiscounted Black-Scholes price, and the learned `Z` at inception approximates `sigma * S0 * delta`.

In [ ]:
torch.manual_seed(13)
problem = BSDEProblem(
    dim=1, x0=100.0, sigma=0.2, maturity=1.0, n_steps=10,
    terminal=lambda x: torch.clamp(x[:, 0] - 100.0, min=0.0),
)
solver = DeepBSDESolver(dim=1, hidden_sizes=(32, 32)).to(DEVICE)
bsde_config = BSDEConfig(n_iterations=800, batch_paths=512, lr=3e-3, seed=7)
bsde_result = train_bsde(problem, solver, bsde_config)
reference = float(bs_call_price(100.0, 100.0, 0.2, 1.0))
print(f"deep BSDE price {bsde_result.y0:.4f}  Black-Scholes {reference:.4f}")
print(f"terminal-matching loss {bsde_result.final_loss:.5f} (a posteriori error certificate)")

## Design notes

Decisions fixed by adversarial verification rather than convention. Antithetic variates are off by default because a good hedge drives residual PnL toward an even function of the noise, where pairing doubles variance exactly when training has converged. Control variates live in evaluation only, since a policy-independent additive control provably does nothing for gradients. The CVaR threshold is a parameter, never a batch quantile. Mixed precision, when it lands, keeps the state in fp32 log space with bf16 confined to network matmuls, because systematic rounding bias survives Monte Carlo averaging while random error does not.

Next phases. A fused CUDA path generator implementing the MarketState, NoiseSpec, and accumulator contracts with a backward that regenerates noise from the Philox offset, then multi-GPU scaling with per-rank noise streams.